# Aula 03 – Classificação de Crédito com Regressão Logística

**Disciplina:** Machine Learning & Modeling  
**Professor:** Samir Ramos  

Nesta aula vamos:

- Entender a diferença entre regressão e classificação.
- Ver o que é regressão **logística** (modelo clássico de classificação binária).
- Treinar um modelo para classificar clientes em **bom pagador (1)** ou **mau pagador (0)**.
- Avaliar o modelo com **accuracy** e **matriz de confusão**.
- Discutir o impacto de errar na aprovação de crédito (visão de negócio).

PASSO 1. Imports e configuração

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

PASSO2. Leitura do CSV dentro do notebook

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

url = "https://raw.githubusercontent.com/samoliverramos/FIAP2026/main/Práticas_Machine_Learning/data/credito_clientes.csv"
df = pd.read_csv(url)

df.head()

PASSO3. EDA rápida

In [ ]:
df.info()

In [ ]:
df.describe()

3.1 - Distribuição da variável alvo:

In [ ]:
sns.countplot(x="bom_pagador", data=df)
plt.title("Distribuição de bons (1) e maus (0) pagadores")
plt.show()

df["bom_pagador"].value_counts(normalize=True)

3.2 - Scatter simples (para visual analógico ao do app de corridas):

In [ ]:
sns.scatterplot(
    x="renda_mensal",
    y="divida_atual",
    hue="bom_pagador",
    data=df,
    alpha=0.6
)
plt.title("Renda vs Dívida atual (colorido por bom_pagador)")
plt.show()

PASSO4. Separando features (X) e alvo (y)

In [ ]:
# Features (entradas)
X = df[["idade", "renda_mensal", "tempo_emprego", "divida_atual", "qtde_atrasos_12m"]]

# Alvo (saída / target)
y = df["bom_pagador"]

X.head()

PASSO5.Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% teste
    random_state=42,
    stratify=y           # mantém proporção de classes
)

print("Tamanho treino:", X_train.shape[0])
print("Tamanho teste :", X_test.shape[0])

PASSO6. Treinando a regressão logística

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

PASSO7.Interpretando os coeficientes

In [ ]:
print("Intercepto:", log_reg.intercept_)

print("\nCoeficientes (na ordem das colunas):")
for col, coef in zip(X.columns, log_reg.coef_[0]):
    print(f"{col:20s}: {coef: .4f}")

COMENTÁRIOS

- Sinal **positivo** do coeficiente → aumenta a chance de ser **bom pagador (1)**.
- Sinal **negativo** → aumenta a chance de ser **mau pagador (0)**.

Exemplo:
- Coeficiente da `renda_mensal` > 0 → quanto maior a renda, maior a probabilidade de bom pagador (faz sentido).
- Coeficiente da `divida_atual` < 0 → quanto maior a dívida, menor a probabilidade de bom pagador.

PASSO8. Avaliação: accuracy (acurácia) em treino e teste

In [ ]:
y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

acc_train = accuracy_score(y_train, y_train_pred)
acc_test = accuracy_score(y_test, y_test_pred)

print(f"Accuracy TREINO: {acc_train:.3f}")
print(f"Accuracy TESTE : {acc_test:.3f}")

PASSO9. Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de confusão\n(0 = mau pagador, 1 = bom pagador)")
plt.show()

MAIS DETALHES....

In [ ]:
print(classification_report(y_test, y_test_pred, target_names=["mau_pagador", "bom_pagador"]))

PASSO 10. Probabilidades em vez de 0/1

In [ ]:
# predict_proba retorna duas colunas: prob de ser 0, prob de ser 1
y_test_proba = log_reg.predict_proba(X_test)
y_test_proba[:5]

Focando na probabilidade de ser bom pagador (classe 1):

In [ ]:
prob_bom = y_test_proba[:, 1]

plt.hist(prob_bom, bins=20, alpha=0.7)
plt.xlabel("Probabilidade de ser bom pagador (classe 1)")
plt.ylabel("Quantidade de clientes")
plt.title("Distribuição das probabilidades no conjunto de teste")
plt.show()

COMENTÁRIOS


- A regressão logística não entrega só 0 ou 1.
- Ela entrega uma **probabilidade** de cada classe (0 ou 1).
- Por padrão, o modelo decide:
  - **Classe 1** se prob(1) ≥ 0.5
  - **Classe 0** se prob(1) < 0.5

Na prática, em crédito:
- Podemos querer ser mais conservadores:
  - Ex.: só aprovar (classe 1) se prob(1) ≥ 0.7.

PASSO11. DESAFIOS PARA OS ALUNOS

## Desafio 1 – Mude o tamanho do conjunto de teste

1. Altere o `test_size` de 0.2 para 0.1 e depois para 0.3 em `train_test_split`.
2. Re-treine o modelo e compare os valores de `Accuracy TREINO` e `Accuracy TESTE`.
3. O modelo ficou mais instável? As métricas variaram muito?

## Desafio 2 – Reduza o número de features

1. Crie uma nova matriz X2 com apenas duas features, por exemplo:
   - `renda_mensal`
   - `qtde_atrasos_12m`
2. Repita o processo: train/test split, treino, avaliação, matriz de confusão.
3. Compare o desempenho com o modelo com todas as features.
   - O modelo piorou muito?
   - Aumentou ou diminuiu o risco de aprovar mau pagador?


PASSO 12. CONCLUSÃO

## Conclusão da Aula 03

Hoje você:

- Viu a diferença prática entre **regressão** (número real) e **classificação** (rótulo 0/1).
- Treinou uma **regressão logística** para um problema realista de **crédito**.
- Usou:
  - `train_test_split` para separar treino e teste.
  - `LogisticRegression` para treinar o modelo.
  - `accuracy_score` e matriz de confusão para avaliar.
- Entendeu que:
  - A regressão logística retorna **probabilidades**.
  - Ajustar o limiar (ex.: 0.5 → 0.7) muda o equilíbrio entre risco e oportunidade.

Na próxima aula, podemos:
- Explorar outra técnica de classificação (KNN, Árvore de decisão ou Random Forest).
- Comparar desempenho com a regressão logística neste mesmo problema de crédito.